# CycleGAN — Python 3.5.4 / TensorFlow 1.4.0 บน Kaggle P100

อัปโหลด ZIP ที่เตรียมให้เป็น **private Dataset** แล้ว Add Input เข้ามาใน notebook นี้
เลือก **GPU P100** และเปิด **Internet** ก่อน Run All

Kaggle ประกาศเลิกให้บริการ P100 วันที่ **15 ก.ย. 2026** หากเหลือ T4 notebook นี้จะหยุดตรวจ
ไม่อัปเกรด TensorFlow หรือเปลี่ยน GPU เงียบ ๆ

ค่าเริ่มต้นเทรน GAN และ classifier อย่างละ 30 epochs ด้วย **loss เดิม** และประเมิน epochs 20/30
ใช้ `MODE = "smoke"` เมื่อต้องการลอง G/D และ classifier อย่างละหนึ่ง update ก่อน
การทดลอง balanced TF2 บนเครื่องส่วนตัวเป็นคนละงานกับ notebook นี้

In [ ]:
import time
from pathlib import Path

SESSION_STARTED = time.monotonic()
MODE = "train"  # "smoke" สำหรับทดสอบก่อน
SESSION_HOURS = 10.0  # รวมเวลา setup; เผื่อจากเพดาน Kaggle 12 ชั่วโมง
CODE_DIR = None  # ปกติค้นจาก Add Input ให้อัตโนมัติ
RESUME_DIR = None  # เช่น "/kaggle/input/previous-output/cyclegan-tf14"
ENV_PREFIX = Path("/kaggle/temp/cyclegan-tf14-env")
OUTPUT = Path("/kaggle/working/cyclegan-tf14")

assert MODE in ("smoke", "train")
assert 0 < SESSION_HOURS <= 10.5

In [ ]:
import subprocess
import sys
import zipfile

gpu = subprocess.check_output([
    "nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"
], text=True).strip()
print(gpu)
if "P100" not in gpu:
    raise RuntimeError("ต้องใช้ GPU P100 สำหรับชุด TF1.4 เดิม หยุดก่อนดาวน์โหลด runtime; ตรวจ Accelerator และประกาศเลิก P100")

if CODE_DIR is None:
    matches = list(Path("/kaggle/input").rglob("bundle-manifest.json"))
    if not matches:
        archives = list(Path("/kaggle/input").rglob("cyclegan-kaggle-tf14.zip"))
        if len(archives) != 1:
            raise RuntimeError("Add Input ที่เป็นแพ็กเกจ cyclegan-kaggle-tf14 หรือระบุ CODE_DIR เอง")
        extracted = Path("/kaggle/temp/cyclegan-upload")
        extracted.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(archives[0]) as archive:
            for name in archive.namelist():
                if "\\" in name or not (extracted / name).resolve().is_relative_to(extracted.resolve()):
                    raise ValueError("Unsafe ZIP member: " + name)
            archive.extractall(extracted)
        matches = list(extracted.rglob("bundle-manifest.json"))
    if len(matches) != 1:
        raise RuntimeError("พบหลาย bundle กรุณาระบุ CODE_DIR ที่ต้องการ")
    CODE_DIR = matches[0].parent
else:
    CODE_DIR = Path(CODE_DIR)
sys.path.insert(0, str(CODE_DIR / "kaggle"))
from session import verify_bundle
from bootstrap import install

manifest = verify_bundle(CODE_DIR)
print("Dataset identity:", manifest["data_identity"])
legacy_python = install(ENV_PREFIX)
print("Legacy Python:", legacy_python)

In [ ]:
remaining = SESSION_HOURS * 3600 - (time.monotonic() - SESSION_STARTED)
if remaining < 600:
    raise RuntimeError("เหลือเวลา session น้อยเกินไปสำหรับเริ่มรัน")
command = [
    sys.executable, str(CODE_DIR / "kaggle/session.py"),
    "--bundle", str(CODE_DIR), "--python", str(legacy_python),
    "--output", str(OUTPUT), "--mode", MODE, "--seconds", str(remaining),
]
if RESUME_DIR:
    command += ["--resume", str(RESUME_DIR)]
result = subprocess.run(command)
print("Runner exit code:", result.returncode)

In [ ]:
import json
from IPython.display import FileLink, display

status_file = OUTPUT / "session-status.json"
if status_file.exists():
    print(json.dumps(json.loads(status_file.read_text()), indent=2, ensure_ascii=False))
archive = OUTPUT.parent / "tf14-results.zip"
if archive.exists():
    display(FileLink(str(archive)))
if result.returncode:
    raise RuntimeError("รันไม่สำเร็จ ตรวจ log และเก็บ output ไว้ก่อนเริ่มใหม่")

## การรันต่อ

ถ้าสถานะเป็น `paused` ให้เก็บ output ของ Saved Version นี้ แล้ว Add Input ของ output นั้น
ใน session ถัดไป ตั้ง `RESUME_DIR` ให้ชี้โฟลเดอร์ที่มี `run-manifest.json` และรันอีกครั้ง
โปรแกรมใช้ code/config/checkpoint ที่บันทึกไว้และคืนสถานะ Adam ด้วย

ดาวน์โหลด `tf14-results.zip` จาก Output ของ Saved Version ได้ด้วย
`smoke_complete` เป็นผลทดสอบสั้น ๆ เท่านั้น; `complete` หมายถึงจบ pipeline ทั้งหมด
ต้องตรวจความเงียบและ note metrics ในรายงานก่อนสรุปผลเทียบ paper